# Semantic Scholar AI Author Coauthor Graph

This notebook builds a coauthor network using Semantic Scholar as the data source.

Definition:

- AI author = an author who published at least one paper in AAAI, IJCAI, ICML, NeurIPS, or ICLR during 2019-2025.

Implementation notes:

1. Use the Semantic Scholar Academic Graph paper bulk search endpoint.
2. Query each target venue and year with conference-type filters.
3. Parse paper authors and build an undirected weighted coauthor graph.
4. Cache API responses locally and retry on rate limiting.

Important caveat:

Semantic Scholar's paper search endpoint is still a search API, not a venue-complete proceedings dump. This notebook is practical and usually good enough for downstream network analysis, but it is not the strongest choice if you need strict completeness guarantees for every paper in each venue-year slice.


In [3]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

START_YEAR = 2019
END_YEAR = 2025
REQUEST_INTERVAL_SECONDS = 1.1
MAX_RETRIES = 6
TIMEOUT = 120
OUTPUT_DIR = Path('data/semantic_scholar_ai_authors_2019_2025')
CACHE_DIR = OUTPUT_DIR / 'http_cache'

# Optional: set your API key in the shell before running the notebook.
S2_API_KEY = os.getenv('S2_API_KEY', '').strip()
BASE_URL = 'https://api.semanticscholar.org/graph/v1'

VENUES = ['AAAI', 'IJCAI', 'ICML', 'NeurIPS', 'ICLR']
VENUE_ALIASES = {
    'AAAI': [
        'AAAI',
        'AAAI Conference on Artificial Intelligence',
        'Proceedings of the AAAI Conference on Artificial Intelligence',
    ],
    'IJCAI': [
        'IJCAI',
        'International Joint Conference on Artificial Intelligence',
    ],
    'ICML': [
        'ICML',
        'International Conference on Machine Learning',
    ],
    'NeurIPS': [
        'NeurIPS',
        'Neural Information Processing Systems',
        'Advances in Neural Information Processing Systems',
    ],
    'ICLR': [
        'ICLR',
        'International Conference on Learning Representations',
    ],
}

# The bulk search endpoint requires a query string.
# This broad query is intended to cover typical AI conference papers.
SEARCH_QUERY = 'artificial intelligence | machine learning | deep learning | neural network | large language model | foundation model | generative ai | natural language processing | computer vision | graph neural network | reinforcement learning | self-supervised learning'

PAPER_FIELDS = ','.join([
    'paperId',
    'title',
    'year',
    'venue',
    'publicationVenue',
    'publicationTypes',
    'authors',
    'url',
])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR


PosixPath('data/semantic_scholar_ai_authors_2019_2025')

In [4]:
LAST_REQUEST_AT = 0.0


def cache_path_for_request(url: str, params: dict | None = None) -> Path:
    payload = json.dumps({'url': url, 'params': params or {}}, sort_keys=True)
    digest = hashlib.sha256(payload.encode('utf-8')).hexdigest()
    return CACHE_DIR / f'{digest}.json'


def throttle_requests(min_interval_seconds: float = REQUEST_INTERVAL_SECONDS) -> None:
    global LAST_REQUEST_AT
    now = time.monotonic()
    wait_seconds = LAST_REQUEST_AT + min_interval_seconds - now
    if wait_seconds > 0:
        time.sleep(wait_seconds)
    LAST_REQUEST_AT = time.monotonic()


def semantic_scholar_get_json(url: str, params: dict | None = None, timeout: int = TIMEOUT, max_retries: int = MAX_RETRIES):
    cache_path = cache_path_for_request(url, params)
    if cache_path.exists():
        print(f'[cache] {url} {params or {}}')
        return json.loads(cache_path.read_text(encoding='utf-8'))

    if params:
        query_string = urllib.parse.urlencode(params, doseq=True)
        full_url = f'{url}?{query_string}'
    else:
        full_url = url

    headers = {
        'User-Agent': 'research-community-analysis/1.0 (academic project; local notebook)',
    }
    if S2_API_KEY:
        headers['x-api-key'] = S2_API_KEY

    req = urllib.request.Request(full_url, headers=headers)

    for attempt in range(max_retries + 1):
        try:
            throttle_requests()
            with urllib.request.urlopen(req, timeout=timeout) as response:
                payload = json.loads(response.read().decode('utf-8'))
            cache_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
            return payload
        except urllib.error.HTTPError as exc:
            if exc.code in {429, 500, 502, 503, 504} and attempt < max_retries:
                retry_after = exc.headers.get('Retry-After') if exc.headers else None
                if retry_after and retry_after.isdigit():
                    wait_seconds = int(retry_after)
                else:
                    wait_seconds = min(120, max(5, int((2 ** attempt) * REQUEST_INTERVAL_SECONDS * 5)))
                print(f'[retry] HTTP {exc.code} for {full_url}; sleeping {wait_seconds}s before retry {attempt + 1}/{max_retries}')
                time.sleep(wait_seconds)
                continue
            error_body = ''
            try:
                error_body = exc.read().decode('utf-8')[:500]
            except Exception:
                pass
            raise RuntimeError(f'HTTP {exc.code} while fetching {full_url}: {error_body}') from exc
        except urllib.error.URLError as exc:
            raise RuntimeError(f'Network error while fetching {full_url}: {exc}') from exc

    raise RuntimeError(f'Exceeded retry limit while fetching {full_url}')


def normalize_author(author: dict) -> tuple[str, str]:
    author_id = str(author.get('authorId') or '').strip()
    author_name = str(author.get('name') or '').strip()
    if author_id:
        return author_id, author_name or author_id
    fallback_key = 'name:' + (author_name.lower() if author_name else 'unknown')
    return fallback_key, author_name or fallback_key


def normalize_text(value: str) -> str:
    return ' '.join(str(value).lower().replace('-', ' ').split())


def venue_matches_requested_venue(paper: dict, requested_venue: str) -> bool:
    aliases = [normalize_text(alias) for alias in VENUE_ALIASES[requested_venue]]
    candidates = [
        normalize_text(paper.get('venue') or ''),
        normalize_text((paper.get('publicationVenue') or {}).get('name') or ''),
    ]

    for candidate in candidates:
        if not candidate:
            continue
        if any(alias == candidate or alias in candidate or candidate in alias for alias in aliases):
            return True
    return False


def normalize_paper(paper: dict, requested_venue: str, requested_year: int) -> dict | None:
    paper_id = str(paper.get('paperId') or '').strip()
    title = str(paper.get('title') or '').strip()

    try:
        year = int(paper.get('year'))
    except (TypeError, ValueError):
        return None

    if year != requested_year:
        return None

    raw_authors = paper.get('authors') or []
    authors = []
    seen = set()
    for author in raw_authors:
        author_key, author_name = normalize_author(author)
        if author_key in seen:
            continue
        seen.add(author_key)
        authors.append({'author_id': author_key, 'author_name': author_name})

    if not authors:
        return None

    venue = str(paper.get('venue') or requested_venue).strip() or requested_venue
    publication_venue = paper.get('publicationVenue') or {}
    publication_venue_name = str(publication_venue.get('name') or '').strip()
    publication_venue_id = str(publication_venue.get('id') or '').strip()

    publication_types = paper.get('publicationTypes') or []

    return {
        'paper_id': paper_id or f'{requested_venue}:{requested_year}:{hashlib.sha1(title.encode("utf-8")).hexdigest()[:16]}',
        'title': title,
        'year': year,
        'venue': venue,
        'requested_venue': requested_venue,
        'publication_venue_name': publication_venue_name,
        'publication_venue_id': publication_venue_id,
        'publication_types': publication_types,
        'url': str(paper.get('url') or '').strip(),
        'authors': authors,
    }


def search_venue_year_papers(venue: str, year: int, query: str = SEARCH_QUERY) -> list[dict]:
    endpoint = f'{BASE_URL}/paper/search/bulk'
    papers = []
    seen_paper_ids = set()

    for venue_alias in VENUE_ALIASES[venue]:
        params = {
            'query': query,
            'fields': PAPER_FIELDS,
            'year': str(year),
            'venue': venue_alias,
            'publicationTypes': 'Conference',
            'sort': 'paperId:asc',
        }

        token = None
        alias_count = 0
        while True:
            page_params = dict(params)
            if token:
                page_params['token'] = token

            response = semantic_scholar_get_json(endpoint, page_params)
            batch = response.get('data') or []

            for raw_paper in batch:
                if not venue_matches_requested_venue(raw_paper, venue):
                    continue
                paper = normalize_paper(raw_paper, requested_venue=venue, requested_year=year)
                if not paper:
                    continue
                if paper['paper_id'] in seen_paper_ids:
                    continue
                seen_paper_ids.add(paper['paper_id'])
                papers.append(paper)
                alias_count += 1

            token = response.get('token')
            if not token:
                break

        print(f'  [alias] {venue} via {venue_alias}: {alias_count} papers')

    return papers


def build_graph(papers: list[dict]):
    author_meta = {}
    edge_weights = Counter()

    for paper in papers:
        for author in paper['authors']:
            author_id = author['author_id']
            author_name = author['author_name']
            if author_id not in author_meta:
                author_meta[author_id] = {
                    'author_name': author_name,
                    'paper_count': 0,
                    'venues': set(),
                    'years': set(),
                }
            author_meta[author_id]['paper_count'] += 1
            author_meta[author_id]['venues'].add(paper['requested_venue'])
            author_meta[author_id]['years'].add(paper['year'])

        author_pairs = sorted((author['author_id'], author['author_name']) for author in paper['authors'])
        for (left_id, _), (right_id, _) in combinations(author_pairs, 2):
            edge_weights[(left_id, right_id)] += 1

    return author_meta, edge_weights


def write_authors_csv(path: Path, author_meta, edge_weights):
    degree_counter = defaultdict(int)
    weighted_degree_counter = defaultdict(int)
    for (left, right), weight in edge_weights.items():
        degree_counter[left] += 1
        degree_counter[right] += 1
        weighted_degree_counter[left] += weight
        weighted_degree_counter[right] += weight

    with path.open('w', newline='', encoding='utf-8') as fh:
        writer = csv.DictWriter(
            fh,
            fieldnames=['author_id', 'author_name', 'paper_count', 'degree', 'weighted_degree', 'venues', 'years'],
        )
        writer.writeheader()
        for author_id in sorted(author_meta):
            meta = author_meta[author_id]
            writer.writerow(
                {
                    'author_id': author_id,
                    'author_name': meta['author_name'],
                    'paper_count': meta['paper_count'],
                    'degree': degree_counter[author_id],
                    'weighted_degree': weighted_degree_counter[author_id],
                    'venues': '|'.join(sorted(meta['venues'])),
                    'years': '|'.join(str(year) for year in sorted(meta['years'])),
                }
            )


def write_edges_csv(path: Path, edge_weights, author_meta):
    with path.open('w', newline='', encoding='utf-8') as fh:
        writer = csv.DictWriter(
            fh,
            fieldnames=['source_author_id', 'source_author_name', 'target_author_id', 'target_author_name', 'weight'],
        )
        writer.writeheader()
        for (source, target), weight in sorted(edge_weights.items(), key=lambda item: (-item[1], item[0][0], item[0][1])):
            writer.writerow(
                {
                    'source_author_id': source,
                    'source_author_name': author_meta[source]['author_name'],
                    'target_author_id': target,
                    'target_author_name': author_meta[target]['author_name'],
                    'weight': weight,
                }
            )


def write_papers_csv(path: Path, papers):
    with path.open('w', newline='', encoding='utf-8') as fh:
        writer = csv.DictWriter(
            fh,
            fieldnames=['paper_id', 'requested_venue', 'venue', 'year', 'title', 'publication_venue_name', 'publication_venue_id', 'publication_types', 'url', 'authors'],
        )
        writer.writeheader()
        for paper in sorted(papers, key=lambda row: (row['year'], row['requested_venue'], row['paper_id'])):
            writer.writerow(
                {
                    'paper_id': paper['paper_id'],
                    'requested_venue': paper['requested_venue'],
                    'venue': paper['venue'],
                    'year': paper['year'],
                    'title': paper['title'],
                    'publication_venue_name': paper['publication_venue_name'],
                    'publication_venue_id': paper['publication_venue_id'],
                    'publication_types': '|'.join(paper['publication_types']),
                    'url': paper['url'],
                    'authors': '|'.join(f"{author['author_id']}::{author['author_name']}" for author in paper['authors']),
                }
            )


def write_graphml(path: Path, author_meta, edge_weights):
    graphml = ET.Element('graphml', xmlns='http://graphml.graphdrawing.org/xmlns')
    ET.SubElement(graphml, 'key', id='label', **{'for': 'node', 'attr.name': 'label', 'attr.type': 'string'})
    ET.SubElement(graphml, 'key', id='paper_count', **{'for': 'node', 'attr.name': 'paper_count', 'attr.type': 'int'})
    ET.SubElement(graphml, 'key', id='venues', **{'for': 'node', 'attr.name': 'venues', 'attr.type': 'string'})
    ET.SubElement(graphml, 'key', id='years', **{'for': 'node', 'attr.name': 'years', 'attr.type': 'string'})
    ET.SubElement(graphml, 'key', id='weight', **{'for': 'edge', 'attr.name': 'weight', 'attr.type': 'int'})

    graph = ET.SubElement(graphml, 'graph', edgedefault='undirected')

    for author_id in sorted(author_meta):
        node = ET.SubElement(graph, 'node', id=author_id)
        ET.SubElement(node, 'data', key='label').text = author_meta[author_id]['author_name']
        ET.SubElement(node, 'data', key='paper_count').text = str(author_meta[author_id]['paper_count'])
        ET.SubElement(node, 'data', key='venues').text = '|'.join(sorted(author_meta[author_id]['venues']))
        ET.SubElement(node, 'data', key='years').text = '|'.join(str(year) for year in sorted(author_meta[author_id]['years']))

    for idx, ((source, target), weight) in enumerate(sorted(edge_weights.items(), key=lambda item: (-item[1], item[0][0], item[0][1])), start=1):
        edge = ET.SubElement(graph, 'edge', id=f'e{idx}', source=source, target=target)
        ET.SubElement(edge, 'data', key='weight').text = str(weight)

    ET.ElementTree(graphml).write(path, encoding='utf-8', xml_declaration=True)


In [5]:
all_papers = []
counts_by_venue_year = {}

for venue in VENUES:
    venue_total = 0
    for year in range(START_YEAR, END_YEAR + 1):
        print(f'[fetch] {venue} {year}')
        papers = search_venue_year_papers(venue=venue, year=year)
        counts_by_venue_year[(venue, year)] = len(papers)
        venue_total += len(papers)
        all_papers.extend(papers)
        print(f'[parse] {venue} {year}: {len(papers)} papers')
    print(f'[done] {venue}: {venue_total} papers\n')

print(f'total papers = {len(all_papers)}')


[fetch] AAAI 2019
  [alias] AAAI via AAAI: 172 papers
  [alias] AAAI via AAAI Conference on Artificial Intelligence: 0 papers
  [alias] AAAI via Proceedings of the AAAI Conference on Artificial Intelligence: 0 papers
[parse] AAAI 2019: 172 papers
[fetch] AAAI 2020
  [alias] AAAI via AAAI: 214 papers
  [alias] AAAI via AAAI Conference on Artificial Intelligence: 0 papers
  [alias] AAAI via Proceedings of the AAAI Conference on Artificial Intelligence: 0 papers
[parse] AAAI 2020: 214 papers
[fetch] AAAI 2021
  [alias] AAAI via AAAI: 269 papers
  [alias] AAAI via AAAI Conference on Artificial Intelligence: 0 papers
  [alias] AAAI via Proceedings of the AAAI Conference on Artificial Intelligence: 0 papers
[parse] AAAI 2021: 269 papers
[fetch] AAAI 2022
  [alias] AAAI via AAAI: 248 papers
  [alias] AAAI via AAAI Conference on Artificial Intelligence: 0 papers
  [alias] AAAI via Proceedings of the AAAI Conference on Artificial Intelligence: 0 papers
[parse] AAAI 2022: 248 papers
[fetch] AAAI

In [6]:
author_meta, edge_weights = build_graph(all_papers)

write_authors_csv(OUTPUT_DIR / 'authors.csv', author_meta, edge_weights)
write_edges_csv(OUTPUT_DIR / 'edges.csv', edge_weights, author_meta)
write_papers_csv(OUTPUT_DIR / 'papers.csv', all_papers)
write_graphml(OUTPUT_DIR / 'graph.graphml', author_meta, edge_weights)

summary = {
    'source': 'Semantic Scholar Academic Graph paper bulk search API',
    'definition': 'AI author = published at least one paper in AAAI, IJCAI, ICML, NeurIPS, or ICLR during 2019-2025',
    'search_query': SEARCH_QUERY,
    'paper_count': len(all_papers),
    'author_count': len(author_meta),
    'edge_count': len(edge_weights),
    'venues': dict(sorted(Counter(p['requested_venue'] for p in all_papers).items())),
    'years': dict(sorted(Counter(p['year'] for p in all_papers).items())),
}

(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'source': 'Semantic Scholar Academic Graph paper bulk search API',
 'definition': 'AI author = published at least one paper in AAAI, IJCAI, ICML, NeurIPS, or ICLR during 2019-2025',
 'search_query': 'artificial intelligence | machine learning | deep learning | neural network | large language model | foundation model | generative ai | natural language processing | computer vision | graph neural network | reinforcement learning | self-supervised learning',
 'paper_count': 4587,
 'author_count': 16969,
 'edge_count': 51557,
 'venues': {'AAAI': 2058, 'ICML': 1738, 'IJCAI': 790, 'NeurIPS': 1},
 'years': {2019: 396,
  2020: 478,
  2021: 575,
  2022: 595,
  2023: 775,
  2024: 881,
  2025: 887}}

In [7]:
top_authors = sorted(
    (
        {
            'author_id': author_id,
            'author_name': meta['author_name'],
            'paper_count': meta['paper_count'],
            'venues': sorted(meta['venues']),
            'years': sorted(meta['years']),
        }
        for author_id, meta in author_meta.items()
    ),
    key=lambda row: (-row['paper_count'], row['author_name'])
)[:20]

top_authors


[{'author_id': '1736651',
  'author_name': 'S. Levine',
  'paper_count': 25,
  'venues': ['ICML'],
  'years': [2019, 2020, 2021, 2022, 2023]},
 {'author_id': '150358650',
  'author_name': 'Zhuoran Yang',
  'paper_count': 23,
  'venues': ['ICML'],
  'years': [2019, 2020, 2021, 2022, 2023, 2025]},
 {'author_id': '1792167',
  'author_name': 'Marcello Restelli',
  'paper_count': 21,
  'venues': ['AAAI', 'ICML', 'IJCAI'],
  'years': [2019, 2020, 2021, 2022, 2023, 2024, 2025]},
 {'author_id': '1689992',
  'author_name': 'P. Abbeel',
  'paper_count': 20,
  'venues': ['AAAI', 'ICML'],
  'years': [2019, 2020, 2021, 2022, 2023]},
 {'author_id': '1712535',
  'author_name': 'Shie Mannor',
  'paper_count': 20,
  'venues': ['AAAI', 'ICML'],
  'years': [2019, 2020, 2021, 2022, 2023]},
 {'author_id': '50218397',
  'author_name': 'Zhaoran Wang',
  'paper_count': 20,
  'venues': ['ICML'],
  'years': [2019, 2020, 2021, 2022, 2023, 2025]},
 {'author_id': '40513470',
  'author_name': 'Jianye Hao',
  'paper

In [8]:
top_edges = sorted(
    (
        {
            'source_author_id': source,
            'source_author_name': author_meta[source]['author_name'],
            'target_author_id': target,
            'target_author_name': author_meta[target]['author_name'],
            'weight': weight,
        }
        for (source, target), weight in edge_weights.items()
    ),
    key=lambda row: (-row['weight'], row['source_author_name'], row['target_author_name'])
)[:20]

top_edges


[{'source_author_id': '150358650',
  'source_author_name': 'Zhuoran Yang',
  'target_author_id': '50218397',
  'target_author_name': 'Zhaoran Wang',
  'weight': 18},
 {'source_author_id': '1721354',
  'source_author_name': 'O. Pietquin',
  'target_author_id': '1737555',
  'target_author_name': 'M. Geist',
  'weight': 12},
 {'source_author_id': '11501567',
  'source_author_name': 'Yunhao Tang',
  'target_author_id': '1708654',
  'target_author_name': 'R. Munos',
  'weight': 12},
 {'source_author_id': '144845456',
  'source_author_name': 'Mark Rowland',
  'target_author_id': '2605877',
  'target_author_name': 'Will Dabney',
  'weight': 10},
 {'source_author_id': '11501567',
  'source_author_name': 'Yunhao Tang',
  'target_author_id': '1806291',
  'target_author_name': 'Michal Valko',
  'weight': 10},
 {'source_author_id': '144845456',
  'source_author_name': 'Mark Rowland',
  'target_author_id': '1708654',
  'target_author_name': 'R. Munos',
  'weight': 9},
 {'source_author_id': '1708654

The exported files are stored in `data/semantic_scholar_ai_authors_2019_2025/`:

- `authors.csv`: author node attributes
- `edges.csv`: coauthor edges and weights
- `papers.csv`: paper list used to build the graph
- `graph.graphml`: ready to import into Gephi / Cytoscape
- `summary.json`: summary statistics

If you have a Semantic Scholar API key, set it before running the notebook for more predictable rate limits:

```bash
export S2_API_KEY='your_key_here'
```
